# California Housing Price Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BurakGuldal10/Machine-Learning-and-Data-Analysis-Projects/blob/main/California_Housing_Price_Regression.ipynb)

## 📌 Proje Özeti / Project Overview
Bu projede, Kaliforniya eyaletindeki konut bloklarına ait sosyo-demografik ve coğrafi özellikler kullanılarak medyan ev fiyatlarının (**MedHouseVal**) tahmin edilmesi hedeflenmiştir. **Doğrusal Regresyon (Linear Regression)**, **Rastgele Orman (Random Forest Regressor)** ve **XGBoost Regressor** modelleri eğitilmiş; özellik mühendisliği (Feature Engineering) ve **GridSearchCV** hiperparametre optimizasyonu ile modelin $R^2$ tahmin gücü artırılmış ve Ortalama Mutlak Hata (MAE) belirgin ölçüde düşürülmüştür.

### 1. Kütüphanelerin Yüklenmesi / Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
print("Kütüphaneler hazır!")

### 2. Veri Setinin Yüklenmesi ve Keşfi / Data Loading & Inspection

In [ ]:
# California Housing veri setini yükleyelim
california = fetch_california_housing(as_frame=True)
df = california.frame

print(f"Veri Seti Boyutu: {df.shape[0]} satır, {df.shape[1]} sütun\n")
display(df.head())

print("\n--- Veri Bilgisi ---")
df.info()

print("\nEksik Değer Sayıları:")
print(df.isnull().sum())

print("\n--- Sayısal Dağılım Özeti ---")
display(df.describe())

### 3. Özellik Mühendisliği / Feature Engineering

In [ ]:
# Hane başına düşen oda, yatak odası oranı ve hane halkı yoğunluğu özellikleri türetelim
df_engineered = df.copy()
df_engineered['Rooms_Per_Household'] = df_engineered['AveRooms'] / df_engineered['AveOccup']
df_engineered['Bedrooms_Per_Room'] = df_engineered['AveBedrms'] / df_engineered['AveRooms']
df_engineered['Population_Per_Household'] = df_engineered['Population'] / df_engineered['AveOccup']

print(f"Yeni özelliklerle veri boyutu: {df_engineered.shape}")
display(df_engineered[['Rooms_Per_Household', 'Bedrooms_Per_Room', 'Population_Per_Household']].head())

### 4. Eğitim ve Test Setine Ayırma / Train-Test Split

In [ ]:
target_col = 'MedHouseVal'
X = df_engineered.drop(columns=[target_col])
y = df_engineered[target_col]

# %80 Eğitim, %20 Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Doğrusal modeller için standartlaştırma
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train boyutu: {X_train.shape}, X_test boyutu: {X_test.shape}")

### 5. Baseline Modeller: Doğrusal Regresyon ve Random Forest

In [ ]:
# 1. Doğrusal Regresyon Baseline
lr_baseline = LinearRegression()
lr_baseline.fit(X_train_scaled, y_train)
y_pred_lr = lr_baseline.predict(X_test_scaled)
lr_r2 = r2_score(y_test, y_pred_lr)
lr_mae = mean_absolute_error(y_test, y_pred_lr)

# 2. Random Forest Regressor Baseline
rf_baseline = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)
y_pred_rf = rf_baseline.predict(X_test)
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)

print(f"Linear Regression (Baseline)  -> R2: {lr_r2:.4f} | MAE: {lr_mae:.4f}")
print(f"Random Forest (Baseline)      -> R2: {rf_r2:.4f} | MAE: {rf_mae:.4f}")

### 6. Gelişmiş Model: XGBoost Regressor & Hiperparametre Optimizasyonu

In [ ]:
# XGBoost Modeli
xgb_model = xgb.XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, subsample=0.8, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

xgb_r2 = r2_score(y_test, y_pred_xgb)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
print(f"XGBoost Regressor (Tuned)     -> R2: {xgb_r2:.4f} | MAE: {xgb_mae:.4f}")

### 7. Tüm Modellerin Karşılaştırma Tablosu / Models Comparison Table

In [ ]:
baseline_rf_r2 = rf_r2
baseline_rf_mae = rf_mae

comparison_df = pd.DataFrame([
    {"Model Aşaması": "Linear Regression (Baseline)", "R2 Score (Tahmin Gücü) ↑": lr_r2, "MAE (Ortalama Hata Payı) ↓": lr_mae},
    {"Model Aşaması": "Random Forest (Baseline)", "R2 Score (Tahmin Gücü) ↑": rf_r2, "MAE (Ortalama Hata Payı) ↓": rf_mae},
    {"Model Aşaması": "XGBoost (GridSearchCV / Nihai Model)", "R2 Score (Tahmin Gücü) ↑": xgb_r2, "MAE (Ortalama Hata Payı) ↓": xgb_mae}
])

comparison_df["Baseline RF'e Göre R2 Gelişimi"] = comparison_df["R2 Score (Tahmin Gücü) ↑"].apply(lambda x: f"{x - baseline_rf_r2:+.6f}")
comparison_df["Baseline RF'e Göre MAE İyileşmesi"] = comparison_df["MAE (Ortalama Hata Payı) ↓"].apply(lambda x: f"{baseline_rf_mae - x:+.6f}")

print("=== TÜM MODELLERİN KARŞILAŞTIRMA TABLOSU ===\n")
display(comparison_df)

# Yan yana karşılaştırma grafiği
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=comparison_df, x='Model Aşaması', y='R2 Score (Tahmin Gücü) ↑', ax=ax[0], palette='Blues_d')
ax[0].set_title('R² Skoru (Yüksek Daha İyi)', fontweight='bold')
ax[0].tick_params(axis='x', rotation=15)

sns.barplot(data=comparison_df, x='Model Aşaması', y='MAE (Ortalama Hata Payı) ↓', ax=ax[1], palette='Reds_d')
ax[1].set_title('MAE Hata Payı (Düşük Daha İyi)', fontweight='bold')
ax[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### 8. Sonuç / Conclusion

- Doğrusal Regresyon modeli ~0.60 seviyesinde bir $R^2$ skoru üretirken, ağaç tabanlı topluluk modelleri (Random Forest ve XGBoost) non-lineer ilişkileri ve coğrafi etkileşimleri yakalayarak belirgin bir üstünlük sağlamıştır.
- Nihai XGBoost modeli en yüksek $R^2$ açıklayıcılık oranı ve en düşük ortalama mutlak hata (MAE) ile konut fiyatı tahmininde en başarılı performansı sergilemiştir.